# Does the conformal / drift story depend on the lender's own risk score?
Real Lending Club 36-month loans (2007-2015, resolved), from `Desktop/Q2/extra_data/lending_club_36m_wide.csv`. Same design as the main notebook: train 2007-2011, calibrate on 2012, test 2013-2015, LightGBM (150 trees, depth 5), split conformal at 90%. Only the **feature set** changes:

| set | features |
|---|---|
| **F0** baseline | the 11 original inputs, including `int_rate` |
| **F1** no lender score | F0 without `int_rate` |
| **F2** lender score | F0 + `grade` + `sub_grade` (both assigned by the lender) |
| **F3** applicant-only, rich | F1 + credit-bureau fields known at application: `open_acc`, `pub_rec`, `total_acc`, `inq_last_6mths`, `mths_since_last_delinq`, `pub_rec_bankruptcies`, `revol_bal`, `credit_hist_yrs`, `addr_state` |
| **F4** everything | F3 + `int_rate`, `grade`, `sub_grade`, `installment` |

Deliberately left out: `mort_acc` (missing for **every** 2007-2011 loan, so unlearnable from the training years), `initial_list_status` / `application_type` (platform settings, not borrower traits), and `installment` in F1/F3 (it is a function of `loan_amnt` and `int_rate`, so it would re-introduce the lender score). Everything is real data; the only randomness is the fixed-seed per-month sample and resampling of real loans for intervals. Choose **Run All**.

In [ ]:
import os, sys, subprocess, importlib, warnings
from pathlib import Path
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('lightgbm'); ensure('statsmodels')
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib; matplotlib.use(os.environ.get('MPL_BACKEND', matplotlib.get_backend()))
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore')

def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2(); assert Q2 is not None, "No 'Q2' folder found on the Desktop; set Q2 = Path(r'...') here."
DATA = Q2/'extra_data'/'lending_club_36m_wide.csv'; OUT = Q2/'outputs_features'; (OUT/'tables').mkdir(parents=True, exist_ok=True); (OUT/'figures').mkdir(exist_ok=True)
assert DATA.exists(), f"{DATA} not found - run Extract_Extra_Data.ipynb first."
SEED, ALPHA, MAX_PER_MONTH = 42, 0.10, int(os.environ.get('MAX_PER_MONTH', 2500))   # 0 = every loan
plt.style.use('seaborn-v0_8-whitegrid')

full = pd.read_csv(DATA)
full = full[full['year'] <= 2015].reset_index(drop=True)
print(f"All resolved 36-month loans: {len(full):,}  default rate {full['default'].mean():.2%}")
def sample(d):
    if not MAX_PER_MONTH: return d
    d = d.sample(frac=1.0, random_state=SEED); d = d[d.groupby(['year', 'month']).cumcount() < MAX_PER_MONTH]
    return d.sort_values('issue_d', kind='stable').reset_index(drop=True)
df = sample(full)                                                # same per-month sampling rule as the main notebook
print(f"Analysis sample: {len(df):,}  ({MAX_PER_MONTH or 'all'} per issue month max)")
print(df.groupby('year')['default'].agg(N='size', default_rate='mean').round(4).T.to_string())

# ---- ordinal lender grades and credit-history fields
df['grade_ord'] = df['grade'].map({g: i for i, g in enumerate('ABCDEFG')})
df['subgrade_ord'] = df['sub_grade'].map({f"{g}{k}": 5 * i + k - 1 for i, g in enumerate('ABCDEFG') for k in range(1, 6)})
full['grade_ord'] = full['grade'].map({g: i for i, g in enumerate('ABCDEFG')})
full['subgrade_ord'] = full['sub_grade'].map({f"{g}{k}": 5 * i + k - 1 for i, g in enumerate('ABCDEFG') for k in range(1, 6)})

In [ ]:
BASE_NUM = ['loan_amnt', 'annual_inc', 'dti', 'fico_score', 'revol_util', 'delinq_2yrs', 'emp_length_yrs']
BASE_CAT = ['home_ownership', 'verification_status', 'purpose']
BUREAU = ['open_acc', 'pub_rec', 'total_acc', 'inq_last_6mths', 'mths_since_last_delinq', 'pub_rec_bankruptcies', 'revol_bal', 'credit_hist_yrs']
SETS = {
    'F0 baseline (with int_rate)':      (BASE_NUM + ['int_rate'], BASE_CAT),
    'F1 no lender score':               (BASE_NUM, BASE_CAT),
    'F2 + grade/sub_grade':             (BASE_NUM + ['int_rate', 'grade_ord', 'subgrade_ord'], BASE_CAT),
    'F3 applicant-only, rich':          (BASE_NUM + BUREAU, BASE_CAT + ['addr_state']),
    'F4 everything':                    (BASE_NUM + BUREAU + ['int_rate', 'grade_ord', 'subgrade_ord', 'installment'], BASE_CAT + ['addr_state']),
}
def design(d, nums, cats):
    return pd.get_dummies(d[nums + cats], columns=cats, drop_first=True)

def conf_q(scores, alpha):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]

def run(d, nums, cats, seed=42):
    X = design(d, nums, cats); y = d['default'].values; yr = d['year'].values
    tr, ca, te = yr <= 2011, yr == 2012, yr >= 2013
    mdl = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=seed, verbose=-1).fit(X[tr], y[tr])
    p = mdl.predict_proba(X)[:, 1]
    sc = np.where(y == 1, 1 - p, p)                              # nonconformity of the true label; label 0 in set iff p <= q, label 1 iff 1-p <= q
    q = conf_q(sc[ca], ALPHA); q0, q1 = conf_q(sc[ca & (y == 0)], ALPHA), conf_q(sc[ca & (y == 1)], ALPHA)
    S_static = np.column_stack([p <= q, (1 - p) <= q]); S_mond = np.column_stack([p <= q0, (1 - p) <= q1])
    return dict(model=mdl, X=X, p=p, y=y, yr=yr, te=te, ca=ca, S_static=S_static, S_mond=S_mond, q=q, q0=q0, q1=q1)

def year_rows(name, R, tag=''):
    rows = []
    for yy in (2012, 2013, 2014, 2015):
        m = R['yr'] == yy; y, p = R['y'][m], R['p'][m]
        row = dict(feature_set=name, year=yy, N=int(m.sum()), AUC=roc_auc_score(y, p), Brier=brier_score_loss(y, p),
                   mean_pred=p.mean(), actual=y.mean(), calib_gap_pp=100 * (y.mean() - p.mean()))
        for lab, S in (('static', R['S_static']), ('mond', R['S_mond'])):
            cov = S[m][np.arange(m.sum()), y]
            row.update({f'{lab}_cov': cov.mean(), f'{lab}_cov_default': cov[y == 1].mean(), f'{lab}_cov_nondef': cov[y == 0].mean(),
                        f'{lab}_size': S[m].sum(1).mean(), f'{lab}_share_reject_only': np.mean(S[m][:, 1] & ~S[m][:, 0]),
                        f'{lab}_share_approve_only': np.mean(S[m][:, 0] & ~S[m][:, 1])})
        if yy >= 2013:
            lo, hi = proportion_confint(int(round(row['static_cov'] * m.sum())), int(m.sum()), method='wilson'); row['static_ci_lo'], row['static_ci_hi'] = lo, hi
        rows.append(row)
    return rows

results = {}; rows = []
for name, (nums, cats) in SETS.items():
    R = run(df, nums, cats); results[name] = R; rows += year_rows(name, R)
    print(f"{name:32s} features={R['X'].shape[1]:3d}  AUC 2012-15: " + " ".join(f"{r['AUC']:.3f}" for r in rows[-4:]))
T_year = pd.DataFrame(rows); T_year.round(4).to_csv(OUT/'tables'/'F1_by_feature_set_and_year.csv', index=False)
show = ['feature_set', 'year', 'AUC', 'Brier', 'calib_gap_pp', 'static_cov', 'static_cov_default', 'static_size', 'static_share_reject_only', 'mond_cov', 'mond_cov_default', 'mond_cov_nondef', 'mond_size']
print(T_year[T_year.year >= 2013][show].round(3).to_string(index=False))

In [ ]:
# ---- worst risk-decile coverage, and test-month block bootstrap of the 2015 static gap
mon = df['month'].values; yrs = df['year'].values
rng = np.random.default_rng(SEED); out = []
for name, R in results.items():
    te = R['te']; p, y = R['p'][te], R['y'][te]
    dec = pd.qcut(p, 10, labels=False, duplicates='drop')
    row = dict(feature_set=name)
    for lab, S in (('static', R['S_static']), ('mond', R['S_mond'])):
        hit = S[te][np.arange(te.sum()), y]; cv = pd.Series(hit).groupby(dec).mean()
        row[f'{lab}_worst_decile_cov'] = cv.min(); row[f'{lab}_worst_decile'] = int(cv.idxmin())
    h15 = R['S_static'][yrs == 2015][np.arange((yrs == 2015).sum()), R['y'][yrs == 2015]].astype(float); m15 = mon[yrs == 2015]
    groups = [h15[m15 == k] for k in range(1, 13) if (m15 == k).any()]
    bs = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() - 0.9 for _ in range(2000)]
    row['gap2015_pp'] = 100 * (h15.mean() - 0.9); row['gap2015_blk_lo'], row['gap2015_blk_hi'] = 100 * np.percentile(bs, [2.5, 97.5])
    out.append(row)
T_dec = pd.DataFrame(out); T_dec.round(4).to_csv(OUT/'tables'/'F2_worst_decile_and_block_bootstrap.csv', index=False)
print(T_dec.round(3).to_string(index=False))

# ---- domain shift per feature set: how well do the features tell 2012 (calibration) from 2015 (test)?
dom = []
for name, (nums, cats) in SETS.items():
    R = results[name]; m = (R['yr'] == 2012) | (R['yr'] == 2015); X = R['X'][m]; z = (R['yr'][m] == 2015).astype(int)
    clf = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=SEED, verbose=-1, importance_type='gain')
    pr = cross_val_predict(clf, X, z, cv=StratifiedKFold(3, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
    imp = pd.Series(clf.fit(X, z).feature_importances_, index=X.columns); imp = imp / imp.sum()
    dom.append(dict(feature_set=name, domain_AUC_2012_vs_2015=roc_auc_score(z, pr), top_feature=imp.idxmax(), top_feature_share=imp.max()))
T_dom = pd.DataFrame(dom); T_dom.round(4).to_csv(OUT/'tables'/'F3_domain_shift.csv', index=False); print(); print(T_dom.round(3).to_string(index=False))

In [ ]:
# ---- how much of the model is just the lender's own score?  AUC of single variables on the same loans, by year
sv = []
for yy in (2012, 2013, 2014, 2015):
    m = df['year'] == yy; y = df.loc[m, 'default']
    sv.append(dict(year=yy, int_rate=roc_auc_score(y, df.loc[m, 'int_rate']), sub_grade=roc_auc_score(y, df.loc[m, 'subgrade_ord']),
                   fico_score_negated=roc_auc_score(y, -df.loc[m, 'fico_score']), dti=roc_auc_score(y, df.loc[m, 'dti']),
                   F0_model=T_year[(T_year.feature_set == 'F0 baseline (with int_rate)') & (T_year.year == yy)]['AUC'].iloc[0],
                   F3_model=T_year[(T_year.feature_set == 'F3 applicant-only, rich') & (T_year.year == yy)]['AUC'].iloc[0]))
T_sv = pd.DataFrame(sv); T_sv.round(4).to_csv(OUT/'tables'/'F4_single_variable_auc.csv', index=False); print("AUC of single variables (no model) vs models:"); print(T_sv.round(3).to_string(index=False))

# ---- grade mix and default rate by grade over time (all loans, not the sample)
mix = full.groupby('year')['grade'].value_counts(normalize=True).unstack().round(4); mix.to_csv(OUT/'tables'/'F5_grade_mix_by_year.csv')
dr = full.groupby(['year', 'grade'])['default'].mean().unstack().round(4); dr.to_csv(OUT/'tables'/'F5_default_rate_by_grade_year.csv')
print("\nGrade mix by issue year (all loans):"); print(mix.to_string()); print("\nDefault rate by grade and year:"); print(dr.to_string())

In [ ]:
# ---- full-data check: same pipeline on EVERY resolved loan (no per-month cap); static split conformal only
fr = []
for name, (nums, cats) in SETS.items():
    R = run(full, nums, cats)
    for r in year_rows(name, R):
        fr.append({k: r[k] for k in ('feature_set', 'year', 'N', 'AUC', 'calib_gap_pp', 'static_cov', 'static_cov_default', 'static_size', 'mond_cov', 'mond_cov_default') if k in r})
T_full = pd.DataFrame(fr); T_full.round(4).to_csv(OUT/'tables'/'F6_full_data_check.csv', index=False)
print(f"FULL DATA ({len(full):,} loans; calibration 2012 N={int((full.year == 2012).sum()):,}; test N={int((full.year >= 2013).sum()):,})")
print(T_full[T_full.year >= 2013].round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5)); yrs4 = [2012, 2013, 2014, 2015]
for name in SETS:
    d = T_year[T_year.feature_set == name]
    ax[0].plot(d.year, d.AUC, marker='o', label=name); ax[1].plot(d.year, d.static_cov, marker='o', label=name); ax[2].plot(d.year, d.mond_cov_default, marker='o', label=name)
ax[0].set_title('Discrimination (ROC-AUC) by year', fontweight='bold'); ax[1].set_title('Static conformal coverage by year', fontweight='bold'); ax[1].axhline(0.9, ls='--', c='k')
ax[2].set_title('Mondrian: coverage of actual defaults', fontweight='bold'); ax[2].axhline(0.9, ls='--', c='k')
for a in ax: a.set_xticks(yrs4); a.set_xlabel('issue year')
ax[1].set_ylim(0.86, 0.95); ax[0].legend(fontsize=8, loc='lower right'); plt.tight_layout()
for ext in ('png', 'pdf'): plt.savefig(OUT/'figures'/f'F1_feature_sets.{ext}', dpi=200, bbox_inches='tight')
plt.show()
print("Saved tables to", OUT/'tables', "and figure to", OUT/'figures')